# LeetCode 407 — Trapping Rain Water II (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def trapRainWater(self, heightMap: List[List[int]]) -> int:

```

Đây là "phiên bản 2 chiều" của bài kinh điển **#42 Trapping Rain Water**. Ở #42, địa hình là một dãy cột xếp thành hàng ngang, nước chỉ có thể chảy sang **trái** hoặc **phải**. Ở bài này, địa hình là một **lưới ô vuông** (như bản đồ địa hình nhìn từ trên xuống), nước có thể chảy theo **4 hướng** và, quan trọng hơn, có thể chảy theo **đường ngoằn ngoèo** để thoát ra ngoài.

Chính chữ "ngoằn ngoèo" đó làm cho mẹo của bài 1D **không áp dụng được trực tiếp**. Notebook này sẽ cho bạn thấy tận mắt vì sao, rồi xây dựng từng bước tới lời giải chuẩn dùng **min-heap** (đống nhỏ nhất).

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc và vẽ một lưới độ cao 2D bằng ASCII, tự đếm tay lượng nước đọng.
2. Ôn lại công thức của bài 1D: `water = min(maxTrái, maxPhải) - h`.
3. **Cách 0 (trực giác, SAI)**: lấy min của max theo 4 hướng thẳng — và một **phản ví dụ chạy được** cho thấy nó sai.
4. Định nghĩa đúng của **mực nước** một ô: "đường thoát nước có điểm cao nhất thấp nhất" (minimax path).
5. **Cách 1 — Mô phỏng mực nước** (giống Bellman-Ford): luôn đúng nhưng chậm, dùng làm **trọng tài** để kiểm tra các cách khác.
6. Kiến thức nền **`heapq`** (min-heap) của Python từ con số 0.
7. **Cách 2 — Min-heap từ biên vào trong** (giống Dijkstra, ý tưởng "bức tường thấp nhất"): cách chính, `O(mn·log(mn))`, có trace từng bước và giải thích invariant.
8. **Cách 3 — Đếm theo lớp nước bằng BFS** (tùy chọn): một góc nhìn khác, học thêm `deque`.
9. Code nộp LeetCode, kiểm thử chéo trên lưới ngẫu nhiên, lỗi hay gặp, mẹo nhận dạng, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Cho một ma trận số nguyên `heightMap` kích thước `m x n`, biểu diễn độ cao của từng ô trong một **bản đồ độ cao 2D** (2D elevation map). Hãy trả về **tổng thể tích nước** mà bản đồ giữ lại được sau khi mưa.

(Mỗi ô có diện tích đáy 1 x 1, nên "thể tích nước trên một ô" chính là "độ dày lớp nước" trên ô đó.)

**Example 1**

```text
Input:  heightMap = [[1,4,3,1,3,2],[3,2,1,3,2,4],[2,3,3,2,3,1]]
Output: 4
Explanation: After the rain, water is trapped between the blocks.
We have two small ponds 1 and 3 units trapped.
The total volume of water trapped is 4.
```

**Example 2**

```text
Input:  heightMap = [[3,3,3,3,3],[3,2,2,2,3],[3,2,1,2,3],[3,2,2,2,3],[3,3,3,3,3]]
Output: 10
```

**Constraints**

- `m == heightMap.length`
- `n == heightMap[i].length`
- `1 <= m, n <= 200`
- `0 <= heightMap[i][j] <= 2 * 10^4`

Nhận xét ngay từ constraints:

- Lưới có tối đa `200 x 200 = 40 000` ô. Một thuật toán `O((mn)^2)` sẽ là khoảng `1.6 * 10^9` phép tính — **quá chậm** với Python. Ta cần cỡ `O(mn log(mn))`.
- `m` hoặc `n` có thể bằng 1 hoặc 2. Khi đó **mọi ô đều nằm ở biên**, nước tràn ra ngoài hết, đáp án là `0`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

example1 = [[1, 4, 3, 1, 3, 2],
            [3, 2, 1, 3, 2, 4],
            [2, 3, 3, 2, 3, 1]]
example2 = [[3, 3, 3, 3, 3],
            [3, 2, 2, 2, 3],
            [3, 2, 1, 2, 3],
            [3, 2, 2, 2, 3],
            [3, 3, 3, 3, 3]]

print("Example 1: m =", len(example1), ", n =", len(example1[0]), "-> kỳ vọng 4")
print("Example 2: m =", len(example2), ", n =", len(example2[0]), "-> kỳ vọng 10")

Example 1: m = 3 , n = 6 -> kỳ vọng 4
Example 2: m = 5 , n = 5 -> kỳ vọng 10


## 2. Vẽ lưới ra cho dễ thấy

Hãy tưởng tượng mỗi số là chiều cao của một **khối gạch** đặt trên ô đó, nhìn từ trên xuống. Mưa đổ xuống khắp nơi. Nước sẽ:

- **chảy khỏi bản đồ** nếu có đường đi xuống thấp dần (hoặc bằng) ra tới mép bản đồ;
- **đọng lại** nếu bị vây kín bởi các khối cao hơn.

Ô ở **biên** (hàng đầu, hàng cuối, cột đầu, cột cuối) không bao giờ giữ nước: nước trên đó tràn thẳng ra ngoài bản đồ.

Hàm `draw_grid` dưới đây in lưới độ cao, và (nếu có) in thêm lưới **nước đọng** bên cạnh. Ô không có nước in dấu `.` cho dễ nhìn.

In [2]:
def draw_grid(heightMap, water=None, title=""):
    """In lưới độ cao; nếu có `water` thì in thêm lưới nước đọng bên phải ('.' = 0)."""
    m, n = len(heightMap), len(heightMap[0])
    values = [v for row in heightMap for v in row]
    if water is not None:
        values += [v for row in water for v in row]
    w = max(len(str(v)) for v in values) + 1          # độ rộng mỗi ô khi in

    def fmt(row, dot_zero=False):
        return "".join(("." if dot_zero and v == 0 else str(v)).rjust(w) for v in row)

    if title:
        print(title)
    gap = " " * 4
    header = "Độ cao".ljust(w * n) + (gap + "Nước đọng" if water is not None else "")
    print(header)
    for i in range(m):
        line = fmt(heightMap[i])
        if water is not None:
            line += gap + fmt(water[i], dot_zero=True)
        print(line)


draw_grid(example1, title="Example 1")

Example 1
Độ cao      
 1 4 3 1 3 2
 3 2 1 3 2 4
 2 3 3 2 3 1


### Đếm tay Example 1

Chỉ có 4 ô **bên trong** (không nằm ở biên): hàng 1, cột 1..4 (đánh số từ 0).

| Ô `(i,j)` | Độ cao | Quan sát | Nước |
|---|---:|---|---:|
| `(1,1)` | 2 | bị vây bởi 4, 3, 3 (ô biên) và ô `(1,2)`; muốn thoát ra phải vượt qua tường cao ít nhất 3 | `3 - 2 = 1` |
| `(1,2)` | 1 | nằm chung "hồ" với `(1,1)`; mặt hồ cao 3 | `3 - 1 = 2` |
| `(1,3)` | 3 | cao bằng mặt hồ, không giữ được gì | `0` |
| `(1,4)` | 2 | bị vây bởi 3, 3, 3, 4 → tường thấp nhất là 3 | `3 - 2 = 1` |

Tổng: `1 + 2 + 0 + 1 = 4`. Đúng như đề nói: một hồ 3 đơn vị (`(1,1)` + `(1,2)`) và một hồ 1 đơn vị (`(1,4)`).

### Đếm tay Example 2

Viền ngoài toàn số 3 (một "bức tường" kín cao 3). Bên trong là 8 ô cao 2 và 1 ô giữa cao 1. Mặt nước dâng tới 3 rồi tràn qua tường:

- 8 ô cao 2: mỗi ô `3 - 2 = 1` → 8.
- Ô giữa cao 1: `3 - 1 = 2`.

Tổng `8 + 2 = 10`.

In [3]:
water_ex1_by_hand = [[0, 0, 0, 0, 0, 0],
                     [0, 1, 2, 0, 1, 0],
                     [0, 0, 0, 0, 0, 0]]
draw_grid(example1, water_ex1_by_hand, "Example 1 (đếm tay)")
print("Tổng =", sum(map(sum, water_ex1_by_hand)))
print()

water_ex2_by_hand = [[0, 0, 0, 0, 0],
                     [0, 1, 1, 1, 0],
                     [0, 1, 2, 1, 0],
                     [0, 1, 1, 1, 0],
                     [0, 0, 0, 0, 0]]
draw_grid(example2, water_ex2_by_hand, "Example 2 (đếm tay)")
print("Tổng =", sum(map(sum, water_ex2_by_hand)))

Example 1 (đếm tay)
Độ cao          Nước đọng
 1 4 3 1 3 2     . . . . . .
 3 2 1 3 2 4     . 1 2 . 1 .
 2 3 3 2 3 1     . . . . . .
Tổng = 4

Example 2 (đếm tay)
Độ cao        Nước đọng
 3 3 3 3 3     . . . . .
 3 2 2 2 3     . 1 1 1 .
 3 2 1 2 3     . 1 2 1 .
 3 2 2 2 3     . 1 1 1 .
 3 3 3 3 3     . . . . .
Tổng = 10


## 3. Ôn lại bài 1D (#42 Trapping Rain Water)

Ở bài 1D, nước trên cột `i` được tính bằng **công thức vàng**:

```text
water[i] = min(maxL[i], maxR[i]) - height[i]
```

- `maxL[i]` = cột cao nhất từ đầu mảng tới `i` (tính cả `i`) — "bức tường bên trái".
- `maxR[i]` = cột cao nhất từ `i` tới cuối mảng — "bức tường bên phải".
- Nước dâng tới **bức tường thấp hơn** trong hai bức tường, vì cao hơn nữa thì tràn qua bên đó.

Ví dụ `height = [4, 2, 0, 3, 2, 5]`, cột `i = 2` (cao 0): `maxL = 4`, `maxR = 5`, nước `= min(4, 5) - 0 = 4`.

Có hai cách tính nhanh mà bạn đã học ở #42:

1. **Prefix max / suffix max**: tính sẵn hai mảng `maxL`, `maxR` → `O(n)` thời gian, `O(n)` bộ nhớ.
2. **Two pointers** (hai con trỏ): bên nào có max nhỏ hơn thì bên đó "đã biết đủ", xử lý cột bên đó rồi tiến vào → `O(n)` thời gian, `O(1)` bộ nhớ.

Ý tưởng "**bên nào thấp hơn thì xử lý bên đó trước**" của two pointers chính là **mầm mống** của lời giải heap cho bài 2D. Hãy nhớ nó!

In [4]:
def trap_1d_prefix_suffix(height: List[int]) -> int:
    n = len(height)
    maxL, maxR = [0] * n, [0] * n
    for i in range(n):
        maxL[i] = max(height[i], maxL[i - 1] if i > 0 else 0)
    for i in range(n - 1, -1, -1):
        maxR[i] = max(height[i], maxR[i + 1] if i < n - 1 else 0)
    return sum(min(maxL[i], maxR[i]) - height[i] for i in range(n))


def trap_1d_two_pointers(height: List[int]) -> int:
    l, r = 0, len(height) - 1
    left_max = right_max = total = 0
    while l < r:
        left_max = max(left_max, height[l])
        right_max = max(right_max, height[r])
        if left_max < right_max:            # bức tường trái thấp hơn -> cột l đã biết đủ
            total += left_max - height[l]
            l += 1
        else:                               # bức tường phải thấp hơn (hoặc bằng)
            total += right_max - height[r]
            r -= 1
    return total


for h in ([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1], [4, 2, 0, 3, 2, 5]):
    print(h, "->", trap_1d_prefix_suffix(h), trap_1d_two_pointers(h))

[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1] -> 6 6
[4, 2, 0, 3, 2, 5] -> 9 9


### Khác biệt cốt lõi giữa 1D và 2D

Trong 1D, nước ở cột `i` chỉ có **đúng 2 lối thoát**: đi thẳng sang trái, hoặc đi thẳng sang phải. Vì vậy `maxL` và `maxR` mô tả **đầy đủ** mọi khả năng thoát nước.

Trong 2D, nước có thể đi theo **bất kỳ đường nào** gồm các bước lên/xuống/trái/phải: đi xuống 1 ô, rẽ phải 2 ô, đi xuống tiếp... Số đường đi là **khổng lồ**. "Bức tường" quanh một ô không còn là 2 con số mà là cả một **đường viền khép kín**.

Ghi nhớ câu này, cả notebook xoay quanh nó: **nước thoát theo đường dễ nhất, và đường dễ nhất có thể ngoằn ngoèo.**

## 4. Kiến thức nền: lưới 2D trong Python

### 4.1. List lồng list

Một lưới `m` hàng, `n` cột trong Python là **list của các list** (list lồng list):

- `grid[i]` là hàng thứ `i` (cũng là một list).
- `grid[i][j]` là ô ở **hàng `i`, cột `j`** (đánh số từ 0).
- `m = len(grid)` (số hàng), `n = len(grid[0])` (số cột của hàng đầu tiên).

### 4.2. Ô biên

Ô `(i, j)` nằm ở **biên** (border) nếu nó ở hàng đầu, hàng cuối, cột đầu hoặc cột cuối:

```python
i == 0 or i == m - 1 or j == 0 or j == n - 1
```

Số ô biên: `2n + 2(m - 2)` khi `m >= 2` (hai hàng trên/dưới mỗi hàng `n` ô, cộng hai cột trái/phải không tính góc). Ví dụ `3 x 6`: `12 + 2 = 14`.

### 4.3. Bốn ô hàng xóm

Từ ô `(i, j)`, bốn ô hàng xóm là `(i-1, j)` (trên), `(i+1, j)` (dưới), `(i, j-1)` (trái), `(i, j+1)` (phải). Ta viết gọn bằng một list **hướng đi** `DIRS`, rồi cộng vào toạ độ. Phải kiểm tra ô mới còn **nằm trong lưới**: `0 <= x < m and 0 <= y < n` (Python cho phép viết so sánh "kẹp" như vậy).

In [5]:
grid = [[1, 4, 3],
        [3, 2, 1]]
m, n = len(grid), len(grid[0])
print("m =", m, ", n =", n)
print("Hàng 1:", grid[1])
print("Ô (1,2):", grid[1][2])          # hàng 1, cột 2 -> 1
print("Ô (0,1):", grid[0][1])          # hàng 0, cột 1 -> 4
print("Ô cuối cùng grid[-1][-1]:", grid[-1][-1])

m = 2 , n = 3
Hàng 1: [3, 2, 1]
Ô (1,2): 1
Ô (0,1): 4
Ô cuối cùng grid[-1][-1]: 1


In [6]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # trên, dưới, trái, phải


def neighbors(i, j, m, n):
    """Liệt kê các ô hàng xóm 4 hướng của (i, j) còn nằm trong lưới m x n."""
    result = []
    for di, dj in DIRS:
        x, y = i + di, j + dj
        if 0 <= x < m and 0 <= y < n:
            result.append((x, y))
    return result


def is_border(i, j, m, n):
    return i == 0 or i == m - 1 or j == 0 or j == n - 1


print("Hàng xóm của (1,1) trong lưới 3x6:", neighbors(1, 1, 3, 6))
print("Hàng xóm của góc (0,0):          ", neighbors(0, 0, 3, 6))

m, n = len(example1), len(example1[0])
border = [(i, j) for i in range(m) for j in range(n) if is_border(i, j, m, n)]
inner = [(i, j) for i in range(m) for j in range(n) if not is_border(i, j, m, n)]
print("Example 1 có", len(border), "ô biên (2n + 2(m-2) =", 2 * n + 2 * (m - 2), ")")
print("Các ô bên trong:", inner)

Hàng xóm của (1,1) trong lưới 3x6: [(0, 1), (2, 1), (1, 0), (1, 2)]
Hàng xóm của góc (0,0):           [(1, 0), (0, 1)]
Example 1 có 14 ô biên (2n + 2(m-2) = 14 )
Các ô bên trong: [(1, 1), (1, 2), (1, 3), (1, 4)]


### 4.4. Bẫy kinh điển khi tạo ma trận: `[[False] * n] * m`

Ta sẽ cần ma trận `visited` (đã thăm hay chưa) cùng kích thước với lưới. Người mới hay viết:

```python
visited = [[False] * n] * m     # SAI!
```

Phép `* m` **không** tạo `m` hàng khác nhau, mà tạo `m` **tham chiếu tới cùng một list**. Sửa một ô ở hàng 0 thì "mọi hàng" đều đổi theo. Cách đúng là dùng **list comprehension** để mỗi hàng là một list mới:

```python
visited = [[False] * n for _ in range(m)]   # ĐÚNG
```

In [7]:
bad = [[False] * 3] * 2
bad[0][1] = True
print("Cách sai :", bad)        # cả 2 hàng đều bị đổi!

good = [[False] * 3 for _ in range(2)]
good[0][1] = True
print("Cách đúng:", good)

Cách sai : [[False, True, False], [False, True, False]]
Cách đúng: [[False, True, False], [False, False, False]]


## 5. Cách 0 — Áp công thức 1D cho 4 hướng (trực giác nhưng SAI)

Ý nghĩ tự nhiên đầu tiên của hầu hết mọi người: "Ở 1D ta lấy min của max trái và max phải. Ở 2D thì lấy min của **4** bức tường: trái, phải, trên, dưới!"

```text
level(i, j) = min(maxLeft, maxRight, maxUp, maxDown)      # mực nước
water(i, j) = level(i, j) - h(i, j)
```

trong đó `maxLeft` là ô cao nhất trên **cùng hàng** từ cột 0 tới cột `j`, `maxUp` là ô cao nhất trên **cùng cột** từ hàng 0 tới hàng `i`, v.v. (đều tính cả chính ô đó, nên `level >= h` và nước không bao giờ âm).

Hãy code thử và chạy trên hai ví dụ của đề.

In [8]:
def trap_four_directions_wrong(heightMap: List[List[int]]) -> int:
    """CẢNH BÁO: cách này SAI trong trường hợp tổng quát. Chỉ dùng để minh họa."""
    m, n = len(heightMap), len(heightMap[0])
    total = 0
    for i in range(m):
        for j in range(n):
            h = heightMap[i][j]
            max_left = max(heightMap[i][:j + 1])
            max_right = max(heightMap[i][j:])
            max_up = max(heightMap[k][j] for k in range(i + 1))
            max_down = max(heightMap[k][j] for k in range(i, m))
            total += min(max_left, max_right, max_up, max_down) - h
    return total


print("Example 1:", trap_four_directions_wrong(example1), "(kỳ vọng 4)")
print("Example 2:", trap_four_directions_wrong(example2), "(kỳ vọng 10)")

Example 1: 4 (kỳ vọng 4)
Example 2: 10 (kỳ vọng 10)


Cả hai ví dụ đều ra đúng! Rất dễ bị lừa và nộp luôn. Nhưng hãy xem lưới sau (gọi là `leak_grid` — lưới "rò nước"):

```text
5 5 5 5
5 1 5 5
5 1 1 5
5 5 1 5      <- ô (3,2) cao 1 nằm ở BIÊN dưới
```

Xét ô `(1,1)` cao 1:

- Cùng hàng 1: bên trái có 5, bên phải có 5, 5 → `maxLeft = 5`, `maxRight = 5`.
- Cùng cột 1: phía trên có 5, phía dưới có 1, 5 → `maxUp = 5`, `maxDown = 5`.
- Công thức 4 hướng: `min(5,5,5,5) - 1 = 4`. Tức là "ô này giữ 4 đơn vị nước".

Nhưng nhìn kỹ: từ `(1,1)` nước chảy xuống `(2,1)` (cao 1), rẽ phải sang `(2,2)` (cao 1), rồi chảy xuống `(3,2)` (cao 1) — **ô biên** — và thoát ra ngoài. Cả đường đi đều cao 1, nên nước không đọng được chút nào! Đáp án đúng của cả lưới là `0`.

Công thức 4 hướng chỉ nhìn **4 đường thẳng**, nó **không thấy đường rẽ góc**.

In [9]:
leak_grid = [[5, 5, 5, 5],
             [5, 1, 5, 5],
             [5, 1, 1, 5],
             [5, 5, 1, 5]]
draw_grid(leak_grid, title="leak_grid (đáp án đúng = 0)")

i, j = 1, 1
row, col = leak_grid[i], [leak_grid[k][j] for k in range(len(leak_grid))]
print(f"\nÔ ({i},{j}): maxLeft={max(row[:j+1])}, maxRight={max(row[j:])}, "
      f"maxUp={max(col[:i+1])}, maxDown={max(col[i:])}")
print("Cách 0 cho tổng nước =", trap_four_directions_wrong(leak_grid), "-> SAI (đúng là 0)")

leak_grid (đáp án đúng = 0)
Độ cao  
 5 5 5 5
 5 1 5 5
 5 1 1 5
 5 5 1 5

Ô (1,1): maxLeft=5, maxRight=5, maxUp=5, maxDown=5
Cách 0 cho tổng nước = 8 -> SAI (đúng là 0)


## 6. Insight: mực nước = "đường thoát có điểm cao nhất thấp nhất"

Hãy định nghĩa lại cho chuẩn. Một **đường thoát** (escape path) của ô `c` là một dãy ô kề nhau (4 hướng) đi từ `c` tới **một ô biên bất kỳ**.

Nước từ `c` đi theo một đường thoát thì phải **vượt qua ô cao nhất** trên đường đó. Gọi đó là **"độ cao đèo"** của đường đi (giống như leo qua một con đèo, khó nhất là đỉnh đèo):

```text
đèo(P) = max { h(v) : v nằm trên đường P }
```

Nước luôn chọn đường **dễ nhất**, tức đường có đèo **thấp nhất**. Vì thế:

```text
level(c) = min over mọi đường thoát P của c  ( max h trên P )
water(c) = level(c) - h(c)
```

Đây gọi là bài toán **minimax path** (đường đi "min của max"). Ví dụ số trong `leak_grid`, ô `(1,1)`:

| Đường thoát | Các độ cao trên đường | Đèo (max) |
|---|---|---:|
| đi thẳng lên `(1,1)→(0,1)` | 1, 5 | 5 |
| đi thẳng trái `(1,1)→(1,0)` | 1, 5 | 5 |
| rẽ góc `(1,1)→(2,1)→(2,2)→(3,2)` | 1, 1, 1, 1 | **1** |

Min của các đèo là `1` → `level = 1` → nước `= 1 - 1 = 0`.

**Vì sao Cách 0 luôn cho kết quả lớn hơn hoặc bằng đáp án đúng?** Bốn đường thẳng chỉ là **4 trong số rất nhiều** đường thoát. Lấy min trên một tập con thì chỉ có thể **lớn hơn hoặc bằng** min trên toàn bộ. Nên Cách 0 **không bao giờ đếm thiếu**, chỉ **đếm thừa** khi đường thoát tốt nhất bị rẽ góc. (Ta sẽ kiểm chứng điều này trên lưới ngẫu nhiên ở mục 11.)

In [10]:
def path_pass_height(grid, path):
    """Độ cao 'đèo' của một đường đi = ô cao nhất trên đường."""
    return max(grid[i][j] for i, j in path)


straight_up = [(1, 1), (0, 1)]
winding = [(1, 1), (2, 1), (2, 2), (3, 2)]
print("Đèo của đường thẳng lên :", path_pass_height(leak_grid, straight_up))
print("Đèo của đường rẽ góc    :", path_pass_height(leak_grid, winding))
print("=> level(1,1) =", min(path_pass_height(leak_grid, p) for p in (straight_up, winding)),
      "(min trên 2 đường này; thật ra phải xét MỌI đường)")

Đèo của đường thẳng lên : 5
Đèo của đường rẽ góc    : 1
=> level(1,1) = 1 (min trên 2 đường này; thật ra phải xét MỌI đường)


## 7. Cách 1 — Mô phỏng mực nước cho tới khi ổn định (giống Bellman-Ford)

Không thể liệt kê mọi đường thoát (quá nhiều). Ta nghĩ theo kiểu **vật lý** thay vì liệt kê:

1. **Ngập hết**: tưởng tượng ban đầu cả bản đồ bị ngập tới độ cao `H` = ô cao nhất của lưới. Không ô nào có mực nước cao hơn `H` được (nước sẽ tràn qua mọi thứ), nên `H` là một **cận trên an toàn**.
2. **Ô biên**: nước tràn ra ngoài, mực nước ô biên = chính độ cao của nó.
3. **Cho nước rút**: với một ô bên trong `c`, mực nước của nó **không thể cao hơn mực nước của hàng xóm thấp nhất** (nếu cao hơn thì nước chảy sang hàng xóm đó), nhưng cũng **không thể thấp hơn mặt đất** `h(c)`. Vậy:

```text
water_level(c) = max( h(c), min(water_level của 4 hàng xóm) )
```

4. Lặp lại bước 3 cho **mọi ô bên trong**, quét đi quét lại toàn lưới, cho tới khi **không còn ô nào thay đổi**.

Ví dụ số: ô cao 1 có 4 hàng xóm với mực nước `5, 5, 1, 5` → `max(1, min(5,5,1,5)) = max(1, 1) = 1`. Ô cao 4 có hàng xóm `3, 3, 3, 5` → `max(4, 3) = 4` (mặt đất cao hơn mặt nước, ô khô).

Lặp "cập nhật cho tới khi ổn định" như thế này giống thuật toán **Bellman-Ford** (thuật toán tìm đường ngắn nhất bằng cách **relax** — "nới lỏng" — các cạnh lặp đi lặp lại). Mực nước chỉ có thể **giảm** sau mỗi lần cập nhật và luôn `>= h`, nên quá trình chắc chắn dừng.

In [11]:
def trap_simulation(heightMap: List[List[int]]) -> int:
    """Cách 1: mô phỏng nước rút dần. Đúng nhưng chậm — dùng làm trọng tài."""
    m, n = len(heightMap), len(heightMap[0])
    H = max(max(row) for row in heightMap)            # cận trên của mọi mực nước
    level = [[heightMap[i][j] if is_border(i, j, m, n) else H for j in range(n)]
             for i in range(m)]
    changed = True
    while changed:                                    # quét tới khi ổn định
        changed = False
        for i in range(1, m - 1):
            for j in range(1, n - 1):
                lowest_nb = min(level[x][y] for x, y in neighbors(i, j, m, n))
                new = max(heightMap[i][j], lowest_nb)
                if new < level[i][j]:
                    level[i][j] = new
                    changed = True
    return sum(level[i][j] - heightMap[i][j] for i in range(m) for j in range(n))


print("Example 1:", trap_simulation(example1))
print("Example 2:", trap_simulation(example2))
print("leak_grid:", trap_simulation(leak_grid))

Example 1: 4
Example 2: 10
leak_grid: 0


### Trace Cách 1 trên `leak_grid`

Ta in **lưới mực nước** sau mỗi lần quét toàn bộ (quét theo thứ tự hàng trên xuống, trái sang phải, cập nhật **ngay tại chỗ** — giá trị mới dùng được luôn cho ô sau trong cùng lần quét). Ô bên trong khởi tạo bằng `H = 5`. Hãy để ý: mỗi lần quét, mặt nước chỉ "rút" được **thêm một ô** dọc theo đường rẽ góc, đi ngược từ lối thoát `(3,2)` vào trong.

In [12]:
def trace_simulation(heightMap):
    m, n = len(heightMap), len(heightMap[0])
    H = max(max(row) for row in heightMap)
    level = [[heightMap[i][j] if is_border(i, j, m, n) else H for j in range(n)]
             for i in range(m)]
    print("Khởi tạo (ô trong = H =", H, "):", level)
    sweep = 0
    changed = True
    while changed:
        changed = False
        sweep += 1
        updates = []
        for i in range(1, m - 1):
            for j in range(1, n - 1):
                new = max(heightMap[i][j], min(level[x][y] for x, y in neighbors(i, j, m, n)))
                if new < level[i][j]:
                    updates.append(f"({i},{j}): {level[i][j]}->{new}")
                    level[i][j] = new
                    changed = True
        print(f"Lần quét {sweep}: cập nhật {updates if updates else 'không có -> DỪNG'}")
    water = [[level[i][j] - heightMap[i][j] for j in range(n)] for i in range(m)]
    draw_grid(heightMap, water, "Kết quả:")
    return sum(map(sum, water))


print("Tổng nước =", trace_simulation(leak_grid))

Khởi tạo (ô trong = H = 5 ): [[5, 5, 5, 5], [5, 5, 5, 5], [5, 5, 5, 5], [5, 5, 1, 5]]
Lần quét 1: cập nhật ['(2,2): 5->1']
Lần quét 2: cập nhật ['(2,1): 5->1']
Lần quét 3: cập nhật ['(1,1): 5->1']
Lần quét 4: cập nhật không có -> DỪNG
Kết quả:
Độ cao      Nước đọng
 5 5 5 5     . . . .
 5 1 5 5     . . . .
 5 1 1 5     . . . .
 5 5 1 5     . . . .
Tổng nước = 0


Tóm tắt trace (khớp với output ở trên):

| Lần quét | Ô thay đổi | Giải thích |
|---:|---|---|
| 1 | `(2,2): 5 → 1` | `(2,2)` kề ô biên `(3,2)` có mực nước 1 → rút xuống `max(1, 1) = 1` |
| 2 | `(2,1): 5 → 1` | giờ `(2,1)` kề `(2,2)` vừa rút xuống 1 |
| 3 | `(1,1): 5 → 1` | `(1,1)` kề `(2,1)` |
| 4 | không có | ổn định → dừng |

Ô `(1,2)` cao 5 giữ nguyên mực 5 (mặt đất bằng mặt nước, khô). Tổng nước = 0. Cách 1 đã "nhìn thấy" đường rẽ góc mà Cách 0 bỏ sót.

### Vì sao Cách 1 đúng? (ý tưởng)

- Mực nước khởi tạo `H` luôn **lớn hơn hoặc bằng** mực nước thật, và mỗi lần cập nhật giữ tính chất này (nếu mọi hàng xóm `>=` mức thật của chúng thì `max(h, min hàng xóm) >=` mức thật của ô).
- Khi ổn định, mỗi ô thỏa `level(c) = max(h(c), min hàng xóm)`. Lần ngược từ `c` theo hàng xóm thấp nhất, ta đi được tới biên qua một đường có đèo đúng bằng `level(c)` → `level(c)` không thể cao hơn mức thật.
- Kẹp cả hai phía → bằng đúng mực nước thật.

### Cách 1 vướng ở đâu?

Thông tin "lối thoát ở đây!" chỉ lan **một ô mỗi lần quét** nếu nó đi ngược chiều quét. Với đường thoát ngoằn ngoèo kiểu **rắn bò** (snake), số lần quét tăng tỉ lệ với độ dài đường đi, mà độ dài này có thể cỡ `mn / 2`.

In [13]:
def make_snake(k):
    """Lưới (2k+1)x(2k+1): tường 9, hành lang cao 1 uốn khúc, lối thoát ở biên dưới."""
    N = 2 * k + 1
    g = [[9] * N for _ in range(N)]
    for r in range(1, N - 1, 2):                    # các hành lang ngang
        for c in range(1, N - 1):
            g[r][c] = 1
    for idx, r in enumerate(range(2, N - 1, 2)):    # các khúc nối, xen kẽ phải/trái
        g[r][N - 2 if idx % 2 == 0 else 1] = 1
    g[N - 1][N - 2 if (k - 1) % 2 == 0 else 1] = 1  # lối thoát ở hàng cuối
    return g


def count_sweeps(heightMap):
    m, n = len(heightMap), len(heightMap[0])
    H = max(max(row) for row in heightMap)
    level = [[heightMap[i][j] if is_border(i, j, m, n) else H for j in range(n)] for i in range(m)]
    sweeps, changed = 0, True
    while changed:
        changed, sweeps = False, sweeps + 1
        for i in range(1, m - 1):
            for j in range(1, n - 1):
                new = max(heightMap[i][j], min(level[x][y] for x, y in neighbors(i, j, m, n)))
                if new < level[i][j]:
                    level[i][j], changed = new, True
    return sweeps


draw_grid(make_snake(3), title="Snake k=3 (nước trong hành lang thoát qua ô 1 ở hàng cuối)")
print()
for k in (3, 6, 12, 20):
    g = make_snake(k)
    N = len(g)
    s = count_sweeps(g)
    print(f"Lưới {N:2}x{N:<2}: {s:4} lần quét, mỗi lần ~{(N-2)**2:4} ô -> ~{s*(N-2)**2:7} lần cập nhật")

Snake k=3 (nước trong hành lang thoát qua ô 1 ở hàng cuối)
Độ cao        
 9 9 9 9 9 9 9
 9 1 1 1 1 1 9
 9 9 9 9 9 1 9
 9 1 1 1 1 1 9
 9 1 9 9 9 9 9
 9 1 1 1 1 1 9
 9 9 9 9 9 1 9

Lưới  7x7 :   14 lần quét, mỗi lần ~  25 ô -> ~    350 lần cập nhật
Lưới 13x13:   42 lần quét, mỗi lần ~ 121 ô -> ~   5082 lần cập nhật
Lưới 25x25:  156 lần quét, mỗi lần ~ 529 ô -> ~  82524 lần cập nhật


Lưới 41x41:  420 lần quét, mỗi lần ~1521 ô -> ~ 638820 lần cập nhật


### Độ phức tạp Cách 1

Đặt `N = m·n` là số ô.

- Mỗi lần quét: `O(N)`.
- Số lần quét: trường hợp xấu `O(N)` (như lưới rắn bò ở trên: lưới `41 x 41` cần hàng trăm lần quét).
- **Time worst-case: `O(N^2)`**. Với `200 x 200` là cỡ `1.6 * 10^9` — Time Limit Exceeded chắc chắn.
- **Space: `O(N)`** cho ma trận mực nước.

Dù chậm, Cách 1 có ưu điểm lớn: **cực kỳ dễ tin là đúng** (nó bám sát vật lý). Ta giữ nó làm **trọng tài** (brute force) để đối chiếu với cách nhanh.

### Mấu chốt để cải tiến

Cách 1 lãng phí vì cập nhật ô theo **thứ tự sai** (thứ tự hàng/cột), nên một ô bị sửa đi sửa lại nhiều lần. Nếu ta xử lý các ô theo **thứ tự mực nước tăng dần** — giống hệt ý tưởng two pointers "xử lý bên thấp hơn trước" ở #42 — thì mỗi ô chỉ cần chốt **đúng một lần**.

Để liên tục lấy ra "ô thấp nhất hiện tại" một cách nhanh chóng, ta cần một cấu trúc dữ liệu tên là **heap**.

## 8. Kiến thức nền: heap và module `heapq`

### Heap là gì?

**Min-heap** (đống nhỏ nhất) là một cấu trúc dữ liệu giống như một "hàng đợi ưu tiên": bạn ném phần tử vào theo thứ tự tùy ý, và lúc nào cũng **lấy ra được phần tử nhỏ nhất** rất nhanh.

| Thao tác | Ý nghĩa | Chi phí (heap có `k` phần tử) |
|---|---|---|
| `heapq.heappush(h, x)` | thêm `x` vào heap `h` | `O(log k)` |
| `heapq.heappop(h)` | lấy ra **và xóa** phần tử nhỏ nhất | `O(log k)` |
| `h[0]` | xem phần tử nhỏ nhất (không xóa) | `O(1)` |
| `heapq.heapify(lst)` | biến một list có sẵn thành heap tại chỗ | `O(k)` |

So sánh: nếu dùng list thường rồi `min(lst)` mỗi lần thì mất `O(k)`; nếu sort lại mỗi lần thêm thì mất `O(k log k)`. Heap cho `O(log k)` cả hai chiều thêm/lấy.

Trong Python, heap chỉ là **một list bình thường**, được module `heapq` sắp xếp theo một quy tắc đặc biệt. Ta khởi tạo bằng `h = []`.

In [14]:
import heapq

h = []
for x in [5, 1, 8, 3, 2]:
    heapq.heappush(h, x)
    print(f"push {x} -> heap (list bên trong) = {h}, nhỏ nhất h[0] = {h[0]}")

print("pop:", heapq.heappop(h), "-> còn", h)
print("pop:", heapq.heappop(h), "-> còn", h)

push 5 -> heap (list bên trong) = [5], nhỏ nhất h[0] = 5
push 1 -> heap (list bên trong) = [1, 5], nhỏ nhất h[0] = 1
push 8 -> heap (list bên trong) = [1, 5, 8], nhỏ nhất h[0] = 1
push 3 -> heap (list bên trong) = [1, 3, 8, 5], nhỏ nhất h[0] = 1
push 2 -> heap (list bên trong) = [1, 2, 8, 5, 3], nhỏ nhất h[0] = 1
pop: 1 -> còn [2, 3, 8, 5]
pop: 2 -> còn [3, 5, 8]


Để ý: list bên trong **không được sắp xếp hoàn toàn** (ví dụ `[1, 2, 8, 5, 3]`). Heap chỉ bảo đảm `h[0]` là nhỏ nhất, và mỗi lần `heappop` nó tự sắp xếp lại một phần (`O(log k)`) để phần tử nhỏ nhất kế tiếp lên đầu. **Đừng** duyệt `for x in h` rồi tưởng là đang đi theo thứ tự tăng dần.

### Heap chứa tuple

Trong bài này, mỗi phần tử của heap sẽ là tuple `(mực_nước, i, j)`. Python so sánh tuple **theo từng thành phần từ trái sang phải**: so phần tử đầu trước; nếu bằng nhau mới so phần tử thứ hai, rồi thứ ba. Vì vậy heap sẽ lấy ra ô có **mực nước nhỏ nhất**; nếu hòa thì ô có `i` nhỏ hơn, rồi `j` nhỏ hơn.

In [15]:
print((3, 0, 5) < (4, 0, 0))     # True: 3 < 4
print((3, 2, 1) < (3, 1, 9))     # False: hòa 3, so tiếp 2 vs 1
print((3, 1, 2) < (3, 1, 9))     # True: hòa 3 và 1, so 2 vs 9

cells_heap = [(4, 0, 1), (1, 0, 0), (3, 1, 0), (1, 0, 3)]
heapq.heapify(cells_heap)        # biến list có sẵn thành heap, O(k)
order = [heapq.heappop(cells_heap) for _ in range(len(cells_heap))]
print("Thứ tự lấy ra:", order)

True
False
True
Thứ tự lấy ra: [(1, 0, 0), (1, 0, 3), (3, 1, 0), (4, 0, 1)]


## 9. Cách 2 — Min-heap từ biên vào trong (cách chính, giống Dijkstra)

### Hình ảnh "bức tường thấp nhất"

Hãy nhìn các ô biên như một **bức tường bao quanh** vùng chưa biết bên trong. Nước bên trong muốn thoát ra phải tràn qua bức tường này, và nó sẽ tràn qua **chỗ thấp nhất** của tường — giống như cái thùng gỗ chỉ chứa được nước tới **thanh gỗ ngắn nhất**.

Gọi chỗ thấp nhất của tường là ô `p`, có mực nước `lv`. Xét một hàng xóm `c` **chưa thăm** của `p` (nằm phía trong tường):

- Nước ở `c` **thoát được qua `p`**, nên mực nước của `c` không vượt quá `max(lv, h(c))`.
- Mọi lối thoát khác của `c` đều phải vượt qua **một chỗ nào đó của tường**, mà mọi chỗ trên tường đều `>= lv` (vì `p` là thấp nhất). Nên mực nước của `c` cũng **không thấp hơn** `max(lv, h(c))`.

Kết luận: **mực nước của `c` chốt luôn = `max(lv, h(c))`**, và nước đọng ở `c` là `max(0, lv - h(c))`.

Sau đó `c` **gia nhập bức tường** với "độ cao tường" mới là `max(lv, h(c))` (nếu `c` có nước thì mặt nước chính là tường; nếu `c` cao hơn `lv` thì chính khối gạch của nó là tường). Còn `p` đã xong việc, rời khỏi tường. Bức tường cứ thế **co dần vào trong** cho tới khi mọi ô đều được thăm.

Liên hệ #42: two pointers chọn bên có max nhỏ hơn để xử lý; ở 2D, bức tường có rất nhiều "bên", nên ta dùng **heap** để luôn lấy ra "bên thấp nhất".

### 9.1. Thuật toán từng bước

1. Tạo `visited` kích thước `m x n` toàn `False`, và một heap rỗng.
2. **Đưa mọi ô biên vào heap** dưới dạng `(h, i, j)` và đánh dấu `visited = True`. (Mực nước ô biên = độ cao của nó.)
3. Lặp khi heap còn phần tử:
   1. `lv, i, j = heappop(heap)` — chỗ thấp nhất của bức tường.
   2. Với mỗi hàng xóm `(x, y)` của `(i, j)` nằm trong lưới và **chưa thăm**:
      - đánh dấu `visited[x][y] = True` **ngay lúc này** (để không bị đưa vào heap lần hai);
      - `total += max(0, lv - h[x][y])`;
      - `heappush(heap, (max(lv, h[x][y]), x, y))`.
4. Trả về `total`.

Tại sao giống **Dijkstra**? Dijkstra cũng lấy ra đỉnh có "khoảng cách" nhỏ nhất từ heap và chốt nó. Ở đây "khoảng cách" không phải tổng độ dài mà là **max độ cao trên đường** (độ cao đèo). Thay phép `+` bằng phép `max`, ta được Dijkstra cho bài minimax path.

In [16]:
import heapq


def trap_heap(heightMap: List[List[int]]) -> int:
    """Cách 2: min-heap từ biên vào trong. O(mn log(mn)) thời gian, O(mn) bộ nhớ."""
    m, n = len(heightMap), len(heightMap[0])
    visited = [[False] * n for _ in range(m)]
    heap = []

    # Bước 2: toàn bộ ô biên là bức tường ban đầu
    for i in range(m):
        for j in range(n):
            if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                heapq.heappush(heap, (heightMap[i][j], i, j))
                visited[i][j] = True

    total = 0
    # Bước 3: luôn xử lý chỗ thấp nhất của bức tường
    while heap:
        lv, i, j = heapq.heappop(heap)
        for di, dj in DIRS:
            x, y = i + di, j + dj
            if 0 <= x < m and 0 <= y < n and not visited[x][y]:
                visited[x][y] = True
                total += max(0, lv - heightMap[x][y])          # nước đọng ở (x, y)
                heapq.heappush(heap, (max(lv, heightMap[x][y]), x, y))  # (x, y) thành tường
    return total


print("Example 1:", trap_heap(example1), "(kỳ vọng 4)")
print("Example 2:", trap_heap(example2), "(kỳ vọng 10)")
print("leak_grid:", trap_heap(leak_grid), "(kỳ vọng 0)")

Example 1: 4 (kỳ vọng 4)
Example 2: 10 (kỳ vọng 10)
leak_grid: 0 (kỳ vọng 0)


### 9.2. Trace Cách 2 trên một lưới "đủ khó"

Hai ví dụ của đề hơi "hiền": mọi ô bên trong đều kề biên hoặc tường đều nhau. Để thấy rõ cả hai nhánh quan trọng (**có nước** và **ô cao hơn mặt nước**), ta dùng lưới `demo_grid`:

```text
5 5 5 5 5
5 1 2 1 5
5 1 4 1 3     <- chỗ thấp nhất của tường: ô biên (2,4) cao 3
5 1 1 1 5
5 5 5 5 5
```

Dự đoán: mặt nước bên trong sẽ là 3 (tràn ra ở `(2,4)`); riêng ô giữa `(2,2)` cao 4 nhô lên khỏi mặt nước. (Cách 0 sẽ đoán sai ở đây vì nhìn thẳng theo hàng 1, cột 2 đều thấy tường 5.)

Hàm `trace_heap` in mỗi lần `pop`; để gọn, mặc định chỉ in các lần pop **thăm được ô mới**. Nó cũng trả về thứ tự thăm và lưới mực nước để vẽ.

In [17]:
def trace_heap(heightMap, show_all=False):
    m, n = len(heightMap), len(heightMap[0])
    visited = [[False] * n for _ in range(m)]
    level = [[None] * n for _ in range(m)]     # mực nước đã chốt của từng ô
    heap = []
    for i in range(m):
        for j in range(n):
            if is_border(i, j, m, n):
                heapq.heappush(heap, (heightMap[i][j], i, j))
                visited[i][j] = True
                level[i][j] = heightMap[i][j]
    print(f"Heap ban đầu: {len(heap)} ô biên, nhỏ nhất = {heap[0]}")
    total, step, popped_levels = 0, 0, []
    while heap:
        lv, i, j = heapq.heappop(heap)
        step += 1
        popped_levels.append(lv)
        actions = []
        for x, y in neighbors(i, j, m, n):
            if not visited[x][y]:
                visited[x][y] = True
                h = heightMap[x][y]
                add = max(0, lv - h)
                total += add
                level[x][y] = max(lv, h)
                heapq.heappush(heap, (level[x][y], x, y))
                actions.append(f"({x},{y}) h={h} +{add}")
        if actions or show_all:
            print(f"#{step:2} pop lv={lv} ({i},{j}) | thăm: {', '.join(actions) or '-'} | total={total}")
    print(f"Tổng {step} lần pop; các lv bị pop có tăng dần? {popped_levels == sorted(popped_levels)}")
    return total, level


demo_grid = [[5, 5, 5, 5, 5],
             [5, 1, 2, 1, 5],
             [5, 1, 4, 1, 3],
             [5, 1, 1, 1, 5],
             [5, 5, 5, 5, 5]]
total_demo, level_demo = trace_heap(demo_grid)
print("Tổng nước demo_grid =", total_demo)

Heap ban đầu: 16 ô biên, nhỏ nhất = (3, 2, 4)
# 1 pop lv=3 (2,4) | thăm: (2,3) h=1 +2 | total=2
# 2 pop lv=3 (2,3) | thăm: (1,3) h=1 +2, (3,3) h=1 +2, (2,2) h=4 +0 | total=6
# 3 pop lv=3 (1,3) | thăm: (1,2) h=2 +1 | total=7
# 4 pop lv=3 (1,2) | thăm: (1,1) h=1 +2 | total=9
# 5 pop lv=3 (1,1) | thăm: (2,1) h=1 +2 | total=11
# 6 pop lv=3 (2,1) | thăm: (3,1) h=1 +2 | total=13
# 7 pop lv=3 (3,1) | thăm: (3,2) h=1 +2 | total=15
Tổng 25 lần pop; các lv bị pop có tăng dần? True
Tổng nước demo_grid = 15


### Đọc trace (khớp output ở trên)

| Bước | Pop `(lv, ô)` | Thăm hàng xóm mới | Nước cộng thêm | Ghi chú |
|---:|---|---|---:|---|
| #1 | `(3, (2,4))` | `(2,3)` h=1 | `3-1 = 2` | chỗ thấp nhất của tường là lối tràn; `(2,3)` vào heap với mức 3 |
| #2 | `(3, (2,3))` | `(1,3)` h=1, `(3,3)` h=1, `(2,2)` h=4 | `2 + 2 + 0` | `(2,2)` cao 4 > 3: **không có nước**, vào heap với mức **4** (chính nó là tường) |
| #3 | `(3, (1,3))` | `(1,2)` h=2 | `1` | |
| #4 | `(3, (1,2))` | `(1,1)` h=1 | `2` | nước "bò" vòng qua ô 4 ở giữa |
| #5 | `(3, (1,1))` | `(2,1)` h=1 | `2` | |
| #6 | `(3, (2,1))` | `(3,1)` h=1 | `2` | |
| #7 | `(3, (3,1))` | `(3,2)` h=1 | `2` | ô cuối cùng bên trong |

Tổng `2 + 4 + 1 + 2 + 2 + 2 + 2 = 15`. Sau đó heap chỉ còn các ô mức 3, 4, 5 không có hàng xóm mới (trace không in các bước "trống" đó).

Hai điều đáng chú ý:

- Các `lv` bị pop ra **không bao giờ giảm** (dòng cuối của output in `True`). Đây là chìa khóa của chứng minh đúng đắn ở mục 9.3.
- Ô `(1,2)` cao 2 nằm ngay cạnh tường 5 ở phía trên, nhưng mực nước của nó là 3 chứ không phải 5 — vì nó được "chạm tới" từ phía lối tràn trước.

In [18]:
water_demo = [[level_demo[i][j] - demo_grid[i][j] for j in range(5)] for i in range(5)]
draw_grid(demo_grid, water_demo, "demo_grid sau khi mưa")
print("\nMực nước đã chốt (level):")
for row in level_demo:
    print(" ".join(str(v) for v in row))
print("\nCách 0 trên demo_grid cho:", trap_four_directions_wrong(demo_grid), "-> sai, đúng là", total_demo)

demo_grid sau khi mưa
Độ cao        Nước đọng
 5 5 5 5 5     . . . . .
 5 1 2 1 5     . 2 1 2 .
 5 1 4 1 3     . 2 . 2 .
 5 1 1 1 5     . 2 2 2 .
 5 5 5 5 5     . . . . .

Mực nước đã chốt (level):
5 5 5 5 5
5 3 3 3 5
5 3 4 3 3
5 3 3 3 5
5 5 5 5 5

Cách 0 trên demo_grid cho: 28 -> sai, đúng là 15


### Trace nhanh trên Example 1 của đề

Ở Example 1, mọi ô bên trong đều kề biên. Hãy để ý ô `(1,3)` cao 3 được thăm **rất sớm** (bước #2, từ ô biên `(0,3)` cao 1) nhưng không có nước, vì `max(1, 3) = 3`. Còn ba ô có nước được thăm khi bức tường đã dâng lên mức 3.

In [19]:
total_ex1, level_ex1 = trace_heap(example1)
water_ex1 = [[level_ex1[i][j] - example1[i][j] for j in range(6)] for i in range(3)]
draw_grid(example1, water_ex1, "Example 1 theo Cách 2")
print("Khớp với đếm tay?", water_ex1 == water_ex1_by_hand)

Heap ban đầu: 14 ô biên, nhỏ nhất = (1, 0, 0)
# 2 pop lv=1 (0,3) | thăm: (1,3) h=3 +0 | total=0
# 7 pop lv=3 (0,2) | thăm: (1,2) h=1 +2 | total=2
# 8 pop lv=3 (0,4) | thăm: (1,4) h=2 +1 | total=3
# 9 pop lv=3 (1,0) | thăm: (1,1) h=2 +1 | total=4
Tổng 18 lần pop; các lv bị pop có tăng dần? True
Example 1 theo Cách 2
Độ cao          Nước đọng
 1 4 3 1 3 2     . . . . . .
 3 2 1 3 2 4     . 1 2 . 1 .
 2 3 3 2 3 1     . . . . . .
Khớp với đếm tay? True


### 9.3. Vì sao Cách 2 đúng? (invariant bằng lời)

Nhắc lại: `level(c)` = min trên mọi đường thoát của `c` của (ô cao nhất trên đường). Ta chứng minh **invariant** (bất biến — điều luôn đúng suốt quá trình chạy):

> **Mỗi ô khi được đưa vào heap thì giá trị đi kèm của nó đúng bằng `level` thật của ô đó.**

**Ô biên**: đường thoát chỉ gồm chính nó → `level = h`. Đúng.

**Ô `c` được thăm từ ô `p` vừa pop với mức `lv`** (giả sử invariant đúng cho mọi ô thăm trước đó):

1. *Không cao hơn*: đi từ `c` sang `p`, rồi theo đường thoát tốt nhất của `p` (đèo = `lv`). Đường này có đèo `= max(h(c), lv)`. Vậy `level(c) <= max(h(c), lv)`.
2. *Không thấp hơn*: lấy một đường thoát bất kỳ của `c`. Đi dọc đường đó, gọi `f` là **ô đã thăm đầu tiên** gặp phải (chắc chắn có, vì đường kết thúc ở biên, mà ô biên đã thăm từ đầu). Ô ngay trước `f` chưa thăm, nên `f` **chưa bị pop** (một ô bị pop thì mọi hàng xóm của nó đã được thăm) → `f` đang nằm trong heap, hoặc `f = p`. Vì `p` là nhỏ nhất của heap, `level(f) >= lv`. Phần đường từ `f` ra biên có đèo `>= level(f) >= lv`, và đường cũng chứa `c` → đèo cả đường `>= max(h(c), lv)`.

Kẹp hai phía: `level(c) = max(h(c), lv)` — chính là giá trị ta đẩy vào heap. Nước tại `c` = `level(c) - h(c) = max(0, lv - h(c))`. Không ô nào bị đếm hai lần nhờ `visited`, không ô nào bị bỏ sót vì lưới liên thông và heap chỉ rỗng khi mọi ô đã thăm.

**Hệ quả**: các mức `lv` bị pop ra không giảm (ô mới vào heap có mức `max(lv, h) >= lv`). Ta kiểm chứng bằng thực nghiệm: so `level` của Cách 2 với mực nước của Cách 1 trên nhiều lưới ngẫu nhiên.

In [20]:
import random


def levels_simulation(heightMap):
    """Trả về lưới mực nước của Cách 1 (để so với level của Cách 2)."""
    m, n = len(heightMap), len(heightMap[0])
    H = max(max(row) for row in heightMap)
    level = [[heightMap[i][j] if is_border(i, j, m, n) else H for j in range(n)] for i in range(m)]
    changed = True
    while changed:
        changed = False
        for i in range(1, m - 1):
            for j in range(1, n - 1):
                new = max(heightMap[i][j], min(level[x][y] for x, y in neighbors(i, j, m, n)))
                if new < level[i][j]:
                    level[i][j], changed = new, True
    return level


def quiet(fn, *args):
    """Chạy hàm in ra màn hình mà không in gì (để test hàng loạt)."""
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args)


random.seed(407)
for _ in range(200):
    m, n = random.randint(1, 6), random.randint(1, 6)
    g = [[random.randint(0, 6) for _ in range(n)] for _ in range(m)]
    _, lv_heap = quiet(trace_heap, g)
    assert lv_heap == levels_simulation(g), g
print("200 lưới ngẫu nhiên: level của Cách 2 == mực nước của Cách 1. Invariant OK!")

200 lưới ngẫu nhiên: level của Cách 2 == mực nước của Cách 1. Invariant OK!


### 9.4. Độ phức tạp Cách 2

Đặt `N = m·n`.

- Mỗi ô được **push đúng 1 lần** và **pop đúng 1 lần** (nhờ đánh dấu `visited` ngay lúc push). Heap không bao giờ quá `N` phần tử.
- Mỗi lần push/pop tốn `O(log N)`; mỗi lần pop duyệt 4 hàng xóm `O(1)`.
- **Time: `O(N log N) = O(mn · log(mn))`**. Với `200 x 200`: khoảng `40 000 * 16 ≈ 640 000` thao tác heap — rất nhanh.
- **Space: `O(N)`** cho `visited` và heap.

(Thực ra heap chỉ chứa "bức tường hiện tại", thường nhỏ hơn `N` nhiều, nhưng trường hợp xấu vẫn là `O(N)`.)

## 10. Cách 3 (tùy chọn) — Đếm theo từng lớp nước bằng BFS

Đây là góc nhìn khác, tương tự cách **monotonic stack** của #42 (đếm nước theo **lớp ngang** thay vì theo cột). Trước hết cần hai kiến thức nền: `deque` và BFS.

### Kiến thức nền: `collections.deque` và BFS (loang)

- `deque` (double-ended queue — hàng đợi hai đầu) cho phép `append` (thêm vào cuối) và `popleft` (lấy ra từ đầu) đều `O(1)`. (Với list thường, `pop(0)` tốn `O(k)` vì phải dịch mọi phần tử.)
- **BFS** (Breadth-First Search — tìm kiếm theo chiều rộng) trên lưới giống như **vết dầu loang**: bắt đầu từ vài ô, mỗi bước lấy một ô ra khỏi hàng đợi và "loang" sang các hàng xóm hợp lệ chưa thăm.

Ví dụ dưới: loang từ các ô biên **thấp hơn 2** trên `leak_grid`, chỉ được đi qua ô thấp hơn 2.

In [21]:
from collections import deque

q = deque([1, 2])
q.append(3)
print("popleft:", q.popleft(), "-> còn", list(q))


def flood_from_border(heightMap, L):
    """BFS: đánh dấu các ô có h < L mà nối được ra biên qua toàn ô có h < L."""
    m, n = len(heightMap), len(heightMap[0])
    seen = [[False] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if is_border(i, j, m, n) and heightMap[i][j] < L:
                seen[i][j] = True
                q.append((i, j))
    while q:
        i, j = q.popleft()
        for x, y in neighbors(i, j, m, n):
            if not seen[x][y] and heightMap[x][y] < L:
                seen[x][y] = True
                q.append((x, y))
    return seen


seen = flood_from_border(leak_grid, 2)
for i, row in enumerate(leak_grid):
    print(" ".join(str(v) for v in row), "   ", " ".join("~" if s else "." for s in seen[i]))
print("'~' = nối được ra biên qua các ô thấp hơn 2 -> các ô 1 đều thoát được nước")

popleft: 1 -> còn [2, 3]
5 5 5 5     . . . .
5 1 5 5     . ~ . .
5 1 1 5     . ~ ~ .
5 5 1 5     . . ~ .
'~' = nối được ra biên qua các ô thấp hơn 2 -> các ô 1 đều thoát được nước


### Ý tưởng đếm theo lớp

Cắt khối nước thành từng **lát ngang** dày 1 đơn vị: lát từ độ cao `L-1` tới `L`. Một ô `c` có nước trong lát này khi và chỉ khi:

- mặt đất `h(c) < L` (ô nằm dưới lát cắt), **và**
- `c` **không** nối được ra biên qua các ô đều thấp hơn `L` (nếu nối được, nước ở lát này chảy mất).

Vậy: `nước trong lát L = số ô có h < L mà BFS từ biên (chỉ qua ô h < L) KHÔNG chạm tới`.

Ví dụ số với Example 2 (độ cao 1, 2, 3):

- Lát `L = 2`: ô có `h < 2` chỉ có ô giữa (cao 1). Biên toàn 3, không loang được → ô giữa giữ nước → **1**.
- Lát `L = 3`: 9 ô bên trong có `h < 3`, biên vẫn không loang được → **9**.
- Lát `L = 4` trở lên: mọi ô đều `< L`, các ô biên loang được khắp nơi → 0.

Tổng `1 + 9 = 10`. ✓

**Tối ưu nhỏ**: mực nước luôn là một độ cao có sẵn trong lưới, nên chỉ cần xét các giá trị `L` là các độ cao **khác nhau** đã sắp xếp `hs[0] < hs[1] < ...`; lát từ `hs[k-1]` tới `hs[k]` dày `hs[k] - hs[k-1]` và có cùng tập ô giữ nước.

In [22]:
def trap_layers(heightMap: List[List[int]], verbose=False) -> int:
    """Cách 3: cộng nước theo từng lớp ngang, mỗi lớp một lần BFS từ biên."""
    m, n = len(heightMap), len(heightMap[0])
    hs = sorted(set(v for row in heightMap for v in row))   # các độ cao khác nhau
    total = 0
    for k in range(1, len(hs)):
        L, thickness = hs[k], hs[k] - hs[k - 1]
        seen = flood_from_border(heightMap, L)
        trapped = sum(1 for i in range(m) for j in range(n)
                      if heightMap[i][j] < L and not seen[i][j])
        total += trapped * thickness
        if verbose:
            print(f"Lớp ({hs[k-1]}, {L}]: dày {thickness}, {trapped} ô giữ nước -> +{trapped * thickness}")
    return total


print("Example 2 =", trap_layers(example2, verbose=True))
print()
print("demo_grid =", trap_layers(demo_grid, verbose=True))

Lớp (1, 2]: dày 1, 1 ô giữ nước -> +1
Lớp (2, 3]: dày 1, 9 ô giữ nước -> +9
Example 2 = 10

Lớp (1, 2]: dày 1, 7 ô giữ nước -> +7
Lớp (2, 3]: dày 1, 8 ô giữ nước -> +8
Lớp (3, 4]: dày 1, 0 ô giữ nước -> +0
Lớp (4, 5]: dày 1, 0 ô giữ nước -> +0
demo_grid = 15


### Độ phức tạp Cách 3

Gọi `D` là số độ cao khác nhau (`D <= min(mn, 20001)`).

- Mỗi lớp một lần BFS `O(mn)` → **Time `O(D · mn)`**. Trường hợp xấu `D` cỡ `2 * 10^4` → `8 * 10^8`: quá chậm cho LeetCode.
- **Space `O(mn)`**.

Cách 3 không dùng để nộp, nhưng rất đáng học vì: (1) nó cho thêm một cách hình dung "nước theo lớp", (2) nó là một **trọng tài thứ hai** hoàn toàn độc lập với Cách 1 và Cách 2, (3) kỹ thuật "BFS loang từ biên" dùng lại ở nhiều bài (ví dụ #130 Surrounded Regions, #417 Pacific Atlantic Water Flow).

## 11. Code nộp LeetCode

Đóng gói Cách 2 vào đúng signature. Một vài chi tiết nhỏ:

- Không cần xử lý riêng `m < 3` hoặc `n < 3`: khi đó mọi ô là ô biên, vòng `while` pop hết mà không thăm thêm ô nào, trả về 0. (Có thể thêm `if m < 3 or n < 3: return 0` để rõ ý, không bắt buộc.)
- Gán `heappush = heapq.heappush` vào biến cục bộ là mẹo nhỏ giúp Python chạy nhanh hơn một chút; không bắt buộc.

In [23]:
import heapq
from typing import List


class Solution:
    def trapRainWater(self, heightMap: List[List[int]]) -> int:
        m, n = len(heightMap), len(heightMap[0])
        if m < 3 or n < 3:
            return 0
        visited = [[False] * n for _ in range(m)]
        heap = []
        for i in range(m):
            for j in range(n):
                if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                    heap.append((heightMap[i][j], i, j))
                    visited[i][j] = True
        heapq.heapify(heap)                      # O(số ô biên)

        total = 0
        while heap:
            lv, i, j = heapq.heappop(heap)       # chỗ thấp nhất của bức tường
            for x, y in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                if 0 <= x < m and 0 <= y < n and not visited[x][y]:
                    visited[x][y] = True
                    h = heightMap[x][y]
                    if h < lv:
                        total += lv - h
                        heapq.heappush(heap, (lv, x, y))
                    else:
                        heapq.heappush(heap, (h, x, y))
        return total


sol = Solution()
tests = [
    (example1, 4),
    (example2, 10),
    (leak_grid, 0),                          # phản ví dụ của Cách 0
    (demo_grid, 15),
    ([[5]], 0),                              # 1 ô
    ([[1, 2, 3, 4]], 0),                     # 1 hàng
    ([[1], [0], [1]], 0),                    # 1 cột
    ([[2, 2], [2, 2]], 0),                   # 2x2: toàn biên
    ([[3, 3, 3], [3, 0, 3], [3, 3, 3]], 3),  # 1 hố đơn giản
    ([[3, 3, 3], [3, 0, 3], [3, 1, 3]], 1),  # hố rò qua ô biên cao 1
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], 0),  # phẳng
    ([[20000] * 3, [20000, 0, 20000], [20000] * 3], 20000),  # giá trị lớn nhất
]
for grid_, expected in tests:
    got = sol.trapRainWater(grid_)
    assert got == expected, (grid_, got, expected)
print(f"{len(tests)} test PASS")

12 test PASS


### Kiểm thử đối chiếu chéo trên lưới ngẫu nhiên

Ta cho 3 cách đúng (Cách 1 mô phỏng, Cách 2 heap / `Solution`, Cách 3 theo lớp) chạy trên **1000 lưới ngẫu nhiên nhỏ** với `random.seed` cố định, và yêu cầu cả ba trùng nhau. Đồng thời đếm xem Cách 0 sai bao nhiêu lần, và kiểm chứng nhận xét ở mục 6: Cách 0 **không bao giờ cho kết quả nhỏ hơn** đáp án đúng.

In [24]:
random.seed(2024)
wrong_count, never_less = 0, True
first_wrong = None
for _ in range(1000):
    m, n = random.randint(1, 7), random.randint(1, 7)
    g = [[random.randint(0, 8) for _ in range(n)] for _ in range(m)]
    a = trap_simulation(g)
    b = trap_heap(g)
    c = trap_layers(g)
    d = sol.trapRainWater(g)
    assert a == b == c == d, (g, a, b, c, d)
    w = trap_four_directions_wrong(g)
    if w != a:
        wrong_count += 1
        if first_wrong is None or len(g) * len(g[0]) < len(first_wrong[0]) * len(first_wrong[0][0]):
            first_wrong = (g, w, a)
    never_less = never_less and (w >= a)

print("1000 lưới: Cách 1 == Cách 2 == Cách 3 == Solution. PASS")
print("Cách 0 sai ở", wrong_count, "/ 1000 lưới; luôn >= đáp án đúng?", never_less)
g, w, a = first_wrong
draw_grid(g, title=f"Phản ví dụ ngẫu nhiên nhỏ nhất tìm được: Cách 0 = {w}, đúng = {a}")

1000 lưới: Cách 1 == Cách 2 == Cách 3 == Solution. PASS
Cách 0 sai ở 272 / 1000 lưới; luôn >= đáp án đúng? True
Phản ví dụ ngẫu nhiên nhỏ nhất tìm được: Cách 0 = 3, đúng = 2
Độ cao
 3 7 4
 2 0 3
 1 0 2
 6 8 6


## 12. So sánh các cách

`N = m·n`, `D` = số độ cao khác nhau.

| Cách | Ý tưởng | Time | Space | Đúng? | Vai trò |
|---|---|---|---|---|---|
| 0. Min của 4 hướng thẳng | áp công thức 1D theo hàng và cột | `O(N·(m+n))` | `O(1)` | **SAI** (đếm thừa khi đường thoát rẽ góc) | bài học về phản ví dụ |
| 1. Mô phỏng nước rút | `level = max(h, min hàng xóm)` lặp tới ổn định | `O(N^2)` worst | `O(N)` | Đúng | trọng tài, dễ tin |
| 2. **Min-heap từ biên** | luôn xử lý chỗ thấp nhất của bức tường | **`O(N log N)`** | `O(N)` | Đúng | **cách nộp bài** |
| 3. Đếm theo lớp + BFS | mỗi lớp ngang: đếm ô không loang ra biên được | `O(D·N)` | `O(N)` | Đúng | góc nhìn khác, trọng tài độc lập |

Mạch tiến hóa: Cách 0 sai vì chỉ xét 4 đường → Cách 1 xét **mọi** đường nhờ lan truyền, nhưng lan truyền theo thứ tự lộn xộn → Cách 2 lan truyền theo **thứ tự mực nước tăng dần** bằng heap, mỗi ô chốt đúng một lần.

## 13. Lỗi người mới hay gặp

**Lỗi 1 — Đẩy `h` thay vì `max(lv, h)` vào heap.**

```python
heapq.heappush(heap, (heightMap[x][y], x, y))     # SAI
heapq.heappush(heap, (max(lv, heightMap[x][y]), x, y))   # ĐÚNG
```

Ô có nước thì "bức tường" tại đó là **mặt nước** (cao `lv`), không phải mặt đất. Đẩy `h` khiến tường bị coi là thấp hơn thực tế, ô phía sau bị tính thiếu nước.

**Lỗi 2 — Đánh dấu `visited` lúc pop nhưng lại cộng nước lúc push.** Một ô có thể bị push nhiều lần từ nhiều hàng xóm khác nhau trước khi được pop → nước bị **cộng nhiều lần**. Quy tắc: *đánh dấu và cộng nước ở cùng một chỗ, ngay lúc thăm lần đầu*.

**Lỗi 3 — Dùng stack (DFS) hoặc queue (BFS) thay cho heap.** Ý tưởng "loang từ biên" chỉ đúng khi luôn lấy ra **chỗ thấp nhất**. Lấy ra ô tùy ý (cuối stack) có thể chốt mực nước của một ô qua một bức tường cao, trong khi còn lối thoát thấp hơn ở nơi khác.

**Lỗi 4 — Quên `max(0, ...)`** khi cộng nước: ô cao hơn mặt nước sẽ cộng số **âm**.

**Lỗi 5 — Chỉ đưa vài ô biên (ví dụ 4 góc) vào heap ban đầu, hoặc coi ô biên có thể giữ nước.** Mọi ô biên đều là lối thoát, phải có mặt trong bức tường từ đầu.

Chạy thử các phiên bản sai trên `demo_grid` (đúng = 15) và Example 2 (đúng = 10):

In [25]:
def heap_variant(heightMap, push_max=True, mark_on_pop=False, use_stack=False, clamp=True):
    """Cách 2 với các 'công tắc' để tái hiện từng lỗi."""
    m, n = len(heightMap), len(heightMap[0])
    visited = [[False] * n for _ in range(m)]
    box = []
    for i in range(m):
        for j in range(n):
            if is_border(i, j, m, n):
                box.append((heightMap[i][j], i, j))
                if not mark_on_pop:
                    visited[i][j] = True
    if not use_stack:
        heapq.heapify(box)
    total = 0
    while box:
        lv, i, j = box.pop() if use_stack else heapq.heappop(box)
        if mark_on_pop:
            if visited[i][j]:
                continue
            visited[i][j] = True
        for x, y in neighbors(i, j, m, n):
            if not visited[x][y]:
                if not mark_on_pop:
                    visited[x][y] = True
                diff = lv - heightMap[x][y]
                total += max(0, diff) if clamp else diff
                item = (max(lv, heightMap[x][y]) if push_max else heightMap[x][y], x, y)
                box.append(item) if use_stack else heapq.heappush(box, item)
    return total


variants = {
    "Đúng (Cách 2)":             dict(),
    "Lỗi 1: push h":             dict(push_max=False),
    "Lỗi 2: visited lúc pop":    dict(mark_on_pop=True),
    "Lỗi 3: dùng stack":         dict(use_stack=True),
    "Lỗi 4: quên max(0, ...)":   dict(clamp=False),
}
for name, kw in variants.items():
    print(f"{name:26} demo_grid = {heap_variant(demo_grid, **kw):3}   example2 = {heap_variant(example2, **kw):3}")

Đúng (Cách 2)              demo_grid =  15   example2 =  10
Lỗi 1: push h              demo_grid =   2   example2 =   2
Lỗi 2: visited lúc pop     demo_grid =  17   example2 =  20
Lỗi 3: dùng stack          demo_grid =  32   example2 =  10
Lỗi 4: quên max(0, ...)    demo_grid =  14   example2 =  10


Giải thích kết quả:

- **Lỗi 1** cho kết quả nhỏ hơn hẳn (Example 2 chỉ còn 2 thay vì 10): ô `(1,1)` cao 2 được thăm từ tường 3 và có nước, nhưng lại vào heap với mức **2** (mặt đất) thay vì 3 (mặt nước). Mức 2 nhỏ hơn mọi ô biên nên nó bị pop ngay, rồi "chốt" các hàng xóm `(1,2)`, `(2,1)` với tường 2 → chúng mất 1 đơn vị nước mỗi ô; lỗi cứ thế lan ra cả hồ.
- **Lỗi 2** cho kết quả lớn hơn: cùng một ô bị cộng nước nhiều lần (mỗi hàng xóm đã pop cộng một lần).
- **Lỗi 3** có thể đúng ở Example 2 (vì tường đều nhau) nhưng sai ở `demo_grid`: stack pop ô biên cao 5 trước, nên nước bị tính dâng tới 5 dù thực ra đã tràn ở `(2,4)` cao 3. Đây là lý do **bắt buộc dùng min-heap**.
- **Lỗi 4** cộng số âm khi gặp ô cao hơn mặt nước: ở `demo_grid`, ô `(2,2)` cao 4 được thăm với `lv = 3` nên cộng `3 - 4 = -1` → 14 thay vì 15. Example 2 không có ô nào nhô lên khỏi mặt nước nên vẫn ra 10 — lại một lỗi mà ví dụ của đề không bắt được.

## 14. Mẹo nhận dạng pattern

- Thấy **"lưới + nước tràn/thoát ra biên"** hoặc **"đường đi mà chi phí là max (hoặc min) trên đường"** → nghĩ ngay tới **minimax path** → **Dijkstra với heap**, thay `+` bằng `max`.
- Thấy công thức 1D dạng `min(max trái, max phải)` mà đề chuyển sang 2D → **cảnh giác**: hướng thẳng không đủ, phải xét đường rẽ.
- Bài "từ biên đi vào" / "từ ngoài đi vào" → khởi tạo heap/queue bằng **toàn bộ ô biên** (multi-source — nhiều nguồn cùng lúc).
- Muốn kiểm tra lời giải tối ưu → viết một bản **mô phỏng chậm nhưng hiển nhiên đúng** rồi đối chiếu trên lưới ngẫu nhiên nhỏ.

## 15. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| #42 Trapping Rain Water | bản 1D; two pointers là "heap 2 phần tử" |
| #11 Container With Most Water | two pointers, min của hai bức tường |
| #778 Swim in Rising Water | minimax path từ góc trên trái tới góc dưới phải — cùng heap (Bài 6 bên dưới) |
| #1631 Path With Minimum Effort | minimax path với chi phí là chênh lệch độ cao |
| #1102 Path With Maximum Minimum Value (Premium) | maximin path — dùng max-heap |
| #417 Pacific Atlantic Water Flow | BFS/DFS loang từ biên, nước chảy xuống |
| #130 Surrounded Regions | loang từ biên để tìm vùng bị vây |
| #743 Network Delay Time | Dijkstra chuẩn (phép `+`) |

Ngoài ra, bài 407 còn có lời giải **Union-Find** (cấu trúc hợp nhất tập hợp): sắp xếp các ô theo độ cao rồi "bật" dần từng ô, gộp với hàng xóm và theo dõi tập nào chạm biên — cũng là một cách nhìn "theo lớp" như Cách 3 nhưng nhanh hơn. Đây là hướng mở rộng thú vị sau khi đã vững Cách 2.

## 16. Bài tập vận dụng

Sáu bài tăng dần độ khó. Hãy tự làm trước, rồi mới xem **Gợi ý theo mức** (mục 17) và **Đáp án** (mục 18).

### Bài 1 — Đoán output (heapq với tuple)

Không chạy code, hãy đoán hai dòng in ra:

```python
import heapq
h = []
for x in [(3, 'a'), (1, 'b'), (3, 'c'), (2, 'd'), (1, 'a')]:
    heapq.heappush(h, x)
print(h[0])
print([heapq.heappop(h) for _ in range(len(h))])
```

### Bài 2 — Tính tay bằng Cách 2

Cho lưới:

```text
3 3 3 3
3 0 2 3
3 1 3 1     <- ô biên (2,3) cao 1
3 3 3 3
```

1. Ô nào bị pop đầu tiên? Nó thăm được ô nào, ô đó được đẩy vào heap với mức bao nhiêu?
2. Lập ma trận mực nước cuối cùng và tính tổng nước.
3. Ô biên `(2,3)` cao 1 rất thấp, vì sao nó **không** làm nước bên trong thoát ra?

### Bài 3 — Điền TODO: viết lại Cách 2

Hoàn thiện hàm theo các bước của mục 9. Đừng quên: đánh dấu `visited` ngay lúc push, và đẩy `max(lv, h)`.

In [26]:
def exercise_trap_heap(heightMap: List[List[int]]) -> int:
    m, n = len(heightMap), len(heightMap[0])
    # TODO 1: tạo visited m x n (cẩn thận bẫy [[False]*n]*m) và heap rỗng
    # TODO 2: đưa mọi ô biên (h, i, j) vào heap, đánh dấu visited
    # TODO 3: while heap: pop (lv, i, j)
    # TODO 4:     với mỗi hàng xóm (x, y) hợp lệ, chưa thăm:
    #                 đánh dấu, cộng max(0, lv - h), push (max(lv, h), x, y)
    # TODO 5: trả về tổng
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_trap_heap(example1) == 4
# assert exercise_trap_heap(example2) == 10
# assert exercise_trap_heap(leak_grid) == 0
# assert exercise_trap_heap(demo_grid) == 15

### Bài 4 — Trả về ma trận mực nước

Viết hàm `exercise_water_matrix(heightMap)` trả về ma trận `water` cùng kích thước, `water[i][j]` = lượng nước đọng trên ô `(i, j)`. Dùng lại Cách 2, chỉ cần ghi lại nước của từng ô thay vì cộng dồn. Có ma trận này rồi bạn có thể gọi `draw_grid(heightMap, water)` để vẽ.

In [27]:
def exercise_water_matrix(heightMap: List[List[int]]) -> List[List[int]]:
    # TODO: như Cách 2, nhưng water[x][y] = max(0, lv - h) thay cho total += ...
    raise NotImplementedError

# Gợi ý test:
# assert exercise_water_matrix(example1) == water_ex1_by_hand
# assert exercise_water_matrix(example2) == water_ex2_by_hand
# assert sum(map(sum, exercise_water_matrix(demo_grid))) == 15

### Bài 5 — Biến thể: đếm số hồ nước

Đề nói Example 1 có "two small ponds" (hai hồ nhỏ). Một **hồ** là một nhóm ô **có nước** (`water > 0`) liên thông với nhau theo 4 hướng. Viết `exercise_count_ponds(heightMap)` trả về số hồ.

Ví dụ: Example 1 → `2`; Example 2 → `1`; `leak_grid` → `0`; lưới dưới đây → `2` (hai hố bị ngăn bởi tường cao 4 ở giữa):

```text
4 4 4 4 4
4 1 4 1 4
4 4 4 4 4
```

In [28]:
def exercise_count_ponds(heightMap: List[List[int]]) -> int:
    # TODO 1: tính ma trận water (dùng lại Bài 4)
    # TODO 2: duyệt mọi ô; gặp ô có water > 0 chưa thăm -> tăng đếm, BFS loang hết hồ đó
    pass

# Gợi ý test:
# assert exercise_count_ponds(example1) == 2
# assert exercise_count_ponds(example2) == 1
# assert exercise_count_ponds(leak_grid) == 0
# assert exercise_count_ponds([[4,4,4,4,4],[4,1,4,1,4],[4,4,4,4,4]]) == 2

### Bài 6 — Mini challenge: LeetCode 778 Swim in Rising Water

Cho lưới `n x n` `grid`, `grid[i][j]` là độ cao ô `(i, j)`. Tại thời điểm `t`, mực nước khắp nơi là `t`; bạn bơi được từ ô này sang ô kề nếu **cả hai ô đều có độ cao `<= t`**. Bơi tức thời. Hỏi thời điểm `t` nhỏ nhất để đi được từ `(0, 0)` tới `(n-1, n-1)`.

Đây đúng là **minimax path**: đáp án = min trên mọi đường từ `(0,0)` tới `(n-1,n-1)` của (ô cao nhất trên đường). Dùng heap giống Cách 2, nhưng **bắt đầu từ một ô** `(0,0)` thay vì cả biên, và **dừng** khi pop ra ô đích.

```text
Input: grid = [[0,2],[1,3]]                    Output: 3
Input: grid = [[0,1,2,3,4],[24,23,22,21,5],[12,13,14,15,16],[11,17,18,19,20],[10,9,8,7,6]]
Output: 16
```

In [29]:
def exercise_swim_in_water(grid: List[List[int]]) -> int:
    # TODO 1: heap = [(grid[0][0], 0, 0)], visited đánh dấu (0, 0)
    # TODO 2: pop (t, i, j); nếu (i, j) là đích -> return t
    # TODO 3: với hàng xóm chưa thăm: đánh dấu, push (max(t, grid[x][y]), x, y)
    pass

# Gợi ý test:
# assert exercise_swim_in_water([[0, 2], [1, 3]]) == 3
# assert exercise_swim_in_water([[0,1,2,3,4],[24,23,22,21,5],[12,13,14,15,16],
#                                [11,17,18,19,20],[10,9,8,7,6]]) == 16

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: heap luôn đưa phần tử **nhỏ nhất** lên `h[0]`.
- Mức 2: tuple so sánh từ trái sang phải; `'a' < 'b'` theo thứ tự chữ cái.
- Mức 3: pop hết heap = sắp xếp tăng dần toàn bộ các tuple.

**Bài 2**
- Mức 1: heap ban đầu gồm 12 ô biên; ô nhỏ nhất là ô cao 1.
- Mức 2: ô biên `(2,3)` chỉ có một hàng xóm bên trong là `(2,2)` cao 3 → `max(1, 3) = 3`.
- Mức 3: sau đó mọi phần tử trong heap đều có mức 3, nên mọi ô bên trong có mực nước 3.

**Bài 3**
- Mức 1: chép lại khung 4 bước ở mục 9, đừng nhìn code.
- Mức 2: `visited = [[False] * n for _ in range(m)]`; điều kiện biên `i == 0 or i == m-1 or j == 0 or j == n-1`.
- Mức 3: trong vòng `for di, dj in DIRS`: `x, y = i + di, j + dj`, kiểm tra `0 <= x < m and 0 <= y < n and not visited[x][y]`.

**Bài 4**
- Mức 1: chỉ khác Bài 3 ở chỗ lưu nước theo ô.
- Mức 2: khởi tạo `water = [[0] * n for _ in range(m)]`; ô biên giữ nguyên 0.
- Mức 3: `water[x][y] = max(0, lv - heightMap[x][y])`.

**Bài 5**
- Mức 1: đây là bài "đếm thành phần liên thông" (connected components) trên ma trận `water`.
- Mức 2: dùng lại khung BFS với `deque` ở mục 10, điều kiện loang là `water[x][y] > 0`.
- Mức 3: hai vòng `for` qua mọi ô; `if water[i][j] > 0 and not seen[i][j]: count += 1; bfs(i, j)`.

**Bài 6**
- Mức 1: "thời điểm nhỏ nhất để đi được" = "đèo thấp nhất" của đường từ đầu tới đích.
- Mức 2: heap lưu `(t, i, j)` với `t` = đèo của đường tốt nhất tới `(i, j)`; giống Cách 2 nhưng chỉ có 1 nguồn.
- Mức 3: khi pop ra `(n-1, n-1)` thì trả về `t` ngay — giống Dijkstra dừng sớm khi chốt đỉnh đích.

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1**: dòng 1 là `(1, 'a')`; dòng 2 là `[(1, 'a'), (1, 'b'), (2, 'd'), (3, 'a'), (3, 'c')]`.

**Bài 2**:

1. Pop đầu tiên: `(1, (2,3))` — ô biên cao 1. Nó thăm `(2,2)` cao 3: nước `max(0, 1-3) = 0`, đẩy vào heap mức `max(1, 3) = 3`.
2. Sau đó mọi mức trong heap là 3 → mực nước cả 4 ô bên trong là 3:

```text
Mực nước        Nước
3 3 3 3         . . . .
3 3 3 3         . 3 1 .
3 3 3 1         . 2 . .
3 3 3 3         . . . .
```

   Tổng = `3 + 1 + 2 + 0 = 6`.

3. Ô thấp `(2,3)` chỉ thông vào trong qua `(2,2)` cao 3; muốn thoát qua đó nước vẫn phải vượt đèo 3. Lỗ thủng trên tường bị "bít" bởi một khối cao phía trong.

Cell dưới kiểm chứng Bài 1, Bài 2 bằng code.

In [30]:
# Kiểm chứng Bài 1
h = []
for x in [(3, 'a'), (1, 'b'), (3, 'c'), (2, 'd'), (1, 'a')]:
    heapq.heappush(h, x)
print(h[0])
print([heapq.heappop(h) for _ in range(len(h))])

# Kiểm chứng Bài 2
grid_b2 = [[3, 3, 3, 3],
           [3, 0, 2, 3],
           [3, 1, 3, 1],
           [3, 3, 3, 3]]
total_b2, level_b2 = trace_heap(grid_b2)
assert total_b2 == 6
assert level_b2 == [[3, 3, 3, 3], [3, 3, 3, 3], [3, 3, 3, 1], [3, 3, 3, 3]]
print("Bài 2: tổng =", total_b2, "PASS")

(1, 'a')
[(1, 'a'), (1, 'b'), (2, 'd'), (3, 'a'), (3, 'c')]
Heap ban đầu: 12 ô biên, nhỏ nhất = (1, 2, 3)
# 1 pop lv=1 (2,3) | thăm: (2,2) h=3 +0 | total=0
# 3 pop lv=3 (0,1) | thăm: (1,1) h=0 +3 | total=3
# 4 pop lv=3 (0,2) | thăm: (1,2) h=2 +1 | total=4
# 7 pop lv=3 (1,1) | thăm: (2,1) h=1 +2 | total=6
Tổng 16 lần pop; các lv bị pop có tăng dần? True
Bài 2: tổng = 6 PASS


In [31]:
# Đáp án Bài 3 và Bài 4
def answer_trap_heap(heightMap: List[List[int]]) -> int:
    m, n = len(heightMap), len(heightMap[0])
    visited = [[False] * n for _ in range(m)]
    heap = []
    for i in range(m):
        for j in range(n):
            if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                heapq.heappush(heap, (heightMap[i][j], i, j))
                visited[i][j] = True
    total = 0
    while heap:
        lv, i, j = heapq.heappop(heap)
        for di, dj in DIRS:
            x, y = i + di, j + dj
            if 0 <= x < m and 0 <= y < n and not visited[x][y]:
                visited[x][y] = True
                total += max(0, lv - heightMap[x][y])
                heapq.heappush(heap, (max(lv, heightMap[x][y]), x, y))
    return total


def answer_water_matrix(heightMap: List[List[int]]) -> List[List[int]]:
    m, n = len(heightMap), len(heightMap[0])
    visited = [[False] * n for _ in range(m)]
    water = [[0] * n for _ in range(m)]
    heap = []
    for i in range(m):
        for j in range(n):
            if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                heapq.heappush(heap, (heightMap[i][j], i, j))
                visited[i][j] = True
    while heap:
        lv, i, j = heapq.heappop(heap)
        for x, y in neighbors(i, j, m, n):
            if not visited[x][y]:
                visited[x][y] = True
                water[x][y] = max(0, lv - heightMap[x][y])
                heapq.heappush(heap, (max(lv, heightMap[x][y]), x, y))
    return water


assert answer_trap_heap(example1) == 4
assert answer_trap_heap(example2) == 10
assert answer_trap_heap(leak_grid) == 0
assert answer_trap_heap(demo_grid) == 15
assert answer_water_matrix(example1) == water_ex1_by_hand
assert answer_water_matrix(example2) == water_ex2_by_hand
assert sum(map(sum, answer_water_matrix(demo_grid))) == 15
draw_grid(demo_grid, answer_water_matrix(demo_grid), "Bài 4 trên demo_grid")
print("Bài 3, Bài 4 PASS")

Bài 4 trên demo_grid
Độ cao        Nước đọng
 5 5 5 5 5     . . . . .
 5 1 2 1 5     . 2 1 2 .
 5 1 4 1 3     . 2 . 2 .
 5 1 1 1 5     . 2 2 2 .
 5 5 5 5 5     . . . . .
Bài 3, Bài 4 PASS


In [32]:
# Đáp án Bài 5 và Bài 6
def answer_count_ponds(heightMap: List[List[int]]) -> int:
    water = answer_water_matrix(heightMap)
    m, n = len(water), len(water[0])
    seen = [[False] * n for _ in range(m)]
    count = 0
    for i in range(m):
        for j in range(n):
            if water[i][j] > 0 and not seen[i][j]:
                count += 1                       # gặp một hồ mới
                seen[i][j] = True
                q = deque([(i, j)])
                while q:                         # BFS loang hết hồ này
                    a, b = q.popleft()
                    for x, y in neighbors(a, b, m, n):
                        if water[x][y] > 0 and not seen[x][y]:
                            seen[x][y] = True
                            q.append((x, y))
    return count


def answer_swim_in_water(grid: List[List[int]]) -> int:
    n = len(grid)
    visited = [[False] * n for _ in range(n)]
    heap = [(grid[0][0], 0, 0)]
    visited[0][0] = True
    while heap:
        t, i, j = heapq.heappop(heap)
        if (i, j) == (n - 1, n - 1):
            return t
        for x, y in neighbors(i, j, n, n):
            if not visited[x][y]:
                visited[x][y] = True
                heapq.heappush(heap, (max(t, grid[x][y]), x, y))
    return -1   # không xảy ra: lưới luôn liên thông


assert answer_count_ponds(example1) == 2
assert answer_count_ponds(example2) == 1
assert answer_count_ponds(leak_grid) == 0
assert answer_count_ponds([[4, 4, 4, 4, 4], [4, 1, 4, 1, 4], [4, 4, 4, 4, 4]]) == 2
assert answer_swim_in_water([[0, 2], [1, 3]]) == 3
assert answer_swim_in_water([[0, 1, 2, 3, 4], [24, 23, 22, 21, 5], [12, 13, 14, 15, 16],
                             [11, 17, 18, 19, 20], [10, 9, 8, 7, 6]]) == 16

random.seed(7)
for _ in range(300):
    m, n = random.randint(1, 6), random.randint(1, 6)
    g = [[random.randint(0, 6) for _ in range(n)] for _ in range(m)]
    assert answer_trap_heap(g) == trap_simulation(g) == sum(map(sum, answer_water_matrix(g)))
print("Bài 5, Bài 6 PASS; đáp án Bài 3/4 khớp trọng tài trên 300 lưới ngẫu nhiên")

Bài 5, Bài 6 PASS; đáp án Bài 3/4 khớp trọng tài trên 300 lưới ngẫu nhiên


## 19. Template ngắn nên nhớ

```python
import heapq

class Solution:
    def trapRainWater(self, heightMap: List[List[int]]) -> int:
        m, n = len(heightMap), len(heightMap[0])
        visited = [[False] * n for _ in range(m)]
        heap = []
        for i in range(m):
            for j in range(n):
                if i in (0, m - 1) or j in (0, n - 1):      # 1) cả biên là bức tường
                    heapq.heappush(heap, (heightMap[i][j], i, j))
                    visited[i][j] = True
        total = 0
        while heap:
            lv, i, j = heapq.heappop(heap)                  # 2) chỗ thấp nhất của tường
            for x, y in ((i-1, j), (i+1, j), (i, j-1), (i, j+1)):
                if 0 <= x < m and 0 <= y < n and not visited[x][y]:
                    visited[x][y] = True                    # 3) đánh dấu ngay khi thăm
                    total += max(0, lv - heightMap[x][y])   # 4) nước = tường - đất
                    heapq.heappush(heap, (max(lv, heightMap[x][y]), x, y))  # 5) thành tường mới
        return total
```

Câu thần chú: **"Cả biên là tường. Luôn lấy chỗ tường thấp nhất. Hàng xóm chưa thăm: nước = max(0, tường − đất), rồi nó thành tường cao max(tường, đất)."**

## 20. Checklist trước khi submit

- Đã đưa **toàn bộ** ô biên (không chỉ 4 góc) vào heap và đánh dấu `visited` chưa?
- Dùng **min-heap** (`heapq`) chứ không phải stack/queue thường?
- Đẩy vào heap `max(lv, h)` chứ không phải `h`?
- Đánh dấu `visited` **ngay lúc push** (để mỗi ô chỉ được cộng nước một lần)?
- Cộng `max(0, lv - h)` (không cộng số âm)?
- Tạo `visited` bằng list comprehension, không dùng `[[False] * n] * m`?
- Đã thử: hai example của đề, lưới 1 hàng / 1 cột / 2x2 (đáp án 0), lưới phẳng, và một lưới "rò nước rẽ góc" như `leak_grid`?
- Nếu nghi ngờ, đã đối chiếu với bản mô phỏng chậm trên vài lưới ngẫu nhiên nhỏ chưa?